# Two-minus sector amplitude $A_n$ — closed form & verification

Deep-water 1D surface waves, sector $\sigma=(-1,-1,+1,\dots,+1)$.

**Canonical closed form** (valid when $|\omega_2|=\min_i|\omega_i|$, e.g. ascending-positive free frequencies):
$$A_n = 2^{\,n-1}\,i\,\omega_1\,\omega_2^{\,2n-5}
      = -2^{\,n-2}\,i\,\frac{\omega_2^{\,2n-5}\,(S_1^2-\omega_2^2+\sum_{i=3}^{n-1}\omega_i^2)}{S_1},\quad S_1=\sum_{j=2}^{n-1}\omega_j.$$

This notebook imports `waterhedron_two_minus.py` (an independent BG port) and checks the formula against the exact Berends-Giele recursion. Tweak the free frequencies and re-run.

In [1]:
from fractions import Fraction as Fr
from waterhedron_two_minus import (make_kinematics, two_minus_sigma,
    bg_amplitude_exact, bg_amplitude, A_canonical, A_canonical_free, A_n5_general)

def im_canonical(ws):
    n = len(ws)
    return Fr(2)**(n-1) * ws[0] * ws[1]**(2*n-5)   # exact Im[A_n]

## 1. Build kinematics and evaluate the formula (n = 5)

In [2]:
n = 5
free_w = [Fr(2), Fr(3), Fr(5)]            # ascending positive  ->  omega_2 = min
ks, ws = make_kinematics(n, free_w, two_minus_sigma(n), Fr(1))
print('omegas =', [str(w) for w in ws])
print('momenta=', [str(k) for k in ks])
print('sum omega =', sum(ws), '   sum sigma*omega^2 =', sum(ks))

bg = bg_amplitude_exact(ks, ws, Fr(1))    # exact BG
print('\nBG (exact)      Im =', bg.im, '  Re =', bg.re)
print('canonical       Im =', im_canonical(ws))
print('free-form (flt)    =', A_canonical_free(n, [float(x) for x in free_w]))
print('match (exact)?     ', bg.re == 0 and bg.im == im_canonical(ws))

omegas = ['-13/2', '2', '3', '5', '-7/2']
momenta= ['-169/4', '-4', '9', '25', '49/4']
sum omega = 0    sum sigma*omega^2 = 0

BG (exact)      Im = -3328   Re = 0
canonical       Im = -3328
free-form (flt)    = -3328j
match (exact)?      True


## 2. Exact verification sweep, n = 4..7

In [3]:
import random
rng = random.Random(1)
for n in (5, 6, 7):
    ok = tot = 0
    for _ in range(200):
        vals = sorted({Fr(rng.randint(1,30), rng.randint(1,5)) for _ in range(n-2)})
        if len(vals) != n-2:
            continue
        ks, ws = make_kinematics(n, list(vals), two_minus_sigma(n), Fr(1))
        mags = [abs(x) for x in ws]
        if min(range(n), key=lambda i: mags[i]) != 1:   # require omega_2 = min
            continue
        bg = bg_amplitude_exact(ks, ws, Fr(1))
        ok += (bg.re == 0 and bg.im == im_canonical(ws)); tot += 1
        if tot >= (40 if n < 7 else 8):
            break
    print(f'n={n}:  exact matches {ok}/{tot}')
print('\nn=4: BGAmplitude is 0/0 (Indeterminate) on the forced n=4 slice;')
print('     the finite limit is A_4 = -8 i w2^3 w3 = 8 i w1 w2^3 (see verify_n4.m).')

n=5:  exact matches 40/40
n=6:  exact matches 40/40
n=7:  exact matches 8/8

n=4: BGAmplitude is 0/0 (Indeterminate) on the forced n=4 slice;
     the finite limit is A_4 = -8 i w2^3 w3 = 8 i w1 w2^3 (see verify_n4.m).


## 3. The amplitude is genuinely chamber-dependent

Outside the canonical chamber ($\omega_2$ no longer the smallest), `BGAmplitude` gives a *different* polynomial. The full $n=5$ rule (`A_n5_general`) handles the generic case; no single rational function covers all chambers.

In [4]:
for fw in ([Fr(2),Fr(3),Fr(5)], [Fr(3),Fr(2),Fr(5)]):   # same set, different order
    ks, ws = make_kinematics(5, fw, two_minus_sigma(5), Fr(1))
    bg = bg_amplitude_exact(ks, ws, Fr(1))
    print(f'free={[str(x) for x in fw]}  BG={bg.im} i   canonical(w2^5)={im_canonical(ws)} i'
          f'   general-rule={A_n5_general([float(x) for x in ws]).imag:.6g} i')

free=['2', '3', '5']  BG=-3328 i   canonical(w2^5)=-3328 i   general-rule=-3328 i
free=['3', '2', '5']  BG=-16128 i   canonical(w2^5)=-23328 i   general-rule=-16128 i
